# Разработка алгоритма BestNodeHalfDiameter

In [ ]:
import networkx as nx
import osmnx as ox

from genesis.core import StateBase, MetricBase, BestPointsBase

In [ ]:
class BestNodesHalfDiameter(BestPointsBase):
    '''
        Поиск лучших узлов графа с использованием алгоритма расчета точки в середине графа. 
        Могут быть возвращены только узлы из которых можно попасть в большую часть других узлов графа.
        Если таковых узлов нет, возвращается ошибка некорректности графа. 
        (граф должен быть проверен на корректность прежде чем будет передан функции)
        
        Узлы для которых метрика не может быть вычислена (например слабо связанные 
        с основным графом) не учитываются. 

        ## Область применения
        Определение размещения одного узла с наилучшими показателями. 
        Дает достаточно точное решение. Хорошо подходит для больших графов.
    '''
    def __init__(self,
                 state_function: StateBase,
                 metric_function: MetricBase,
                 weight='travel_time',
                 appr_val = 0.95,
                 **kwargs) -> None:
        self.appr_val = appr_val
        super().__init__(state_function, metric_function, **kwargs)

    def __call__(self, env:nx.MultiDiGraph, **kwargs):
        '''
        
        '''
        # 1 Выбираем произвольную точку. По-умолчанию берем просто первую из списка узлов
        nd = list(env.nodes())[0]
        # 2 Находим самую отдаленную от нее (входящую) -- периферия №1
        lngs = nx.shortest_path_length(env, target=nd, weight='travel_time')
        corner_1 = max(lngs, key=lngs.get)
        # 2 Находим самую отдаленную от нее (входящую) -- периферия №2
        lngs = nx.shortest_path_length(env, target=corner_1, weight='travel_time')
        corner_2 = max(lngs, key=lngs.get)
        # # 3 Находим самую отдаленную от нее (входящую) -- периферия №1
        # lngs = nx.shortest_path_length(env, target=corner_2, weight='travel_time')
        # corner_1 = max(lngs, key=lngs.get)

        # ! Добавить расчет маршрута !
        diameter = nx.shortest_path_length(env, corner_1, corner_2, weight='travel_time')

        # Находим точку примерно по середине диаметра